In [1]:
from pyspark.sql import SparkSession
import pyspark
print(pyspark.__version__)
from pyspark.sql import functions as F

3.4.1


In [2]:
spark = (SparkSession.builder
    .appName("preprocess-stock")
    .master("spark://spark-master:7077")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.hadoop.hive.exec.dynamic.partition", "true")
    .config("spark.hadoop.hive.exec.dynamic.partition.mode", "nonstrict")
    .getOrCreate())

spark.sql("USE DATABASE CryptoPredictions")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/24 16:02:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/24 16:02:28 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

In [27]:
folder = "hdfs://namenode:8020/nifi/stock-prices"
from pyspark.sql.types import LongType

# Wczytanie całego folderu do DataFrame
df_all = spark.read.parquet(folder)

# Spark zapisuje plik źródłowy w kolumnie __file__ (od Spark 3.x)
df_with_file = df_all.withColumn("__file__", F.input_file_name())

# Pobranie schematów per plik
files = df_with_file.select("__file__").distinct().collect()

for row in files:
    file_path = row["__file__"]
    df_file = spark.read.parquet(file_path)
    dtype = df_file.schema["lastVolume"].dataType
    if not isinstance(dtype, LongType):
        print(f"{file_path} -> lastVolume: {dtype}")

In [28]:
# folder = "hdfs://namenode:8020/nifi/stock-prices"
# bad_file = "hdfs://namenode:8020/nifi/stock-prices/stock_prices_2025-11-18-14.parquet"

# # Read all parquet files
# df_all = spark.read.parquet(folder)

# # Filter out the problematic file
# df = df_all.withColumn("__file__", F.input_file_name()) \
#                     .filter(F.col("__file__") != bad_file) \
#                     .drop("__file__")
# df.show(1, vertical=True, truncate=False)
# df.printSchema()

In [5]:
folder = "hdfs://namenode:8020/nifi/stock-prices"
df = spark.read.parquet(folder)

In [3]:
def transform_index_snapshot(df):
    res = (
        df
        # Dopasowanie nazw
        .withColumn("IndexName", F.col("exchange"))
        .withColumn("Datetime", F.to_timestamp("fetch_timestamp"))
        .withColumn("CurrentPrice", F.col("lastPrice"))
        .withColumn("CurrentVolume", F.col("lastVolume"))
        .withColumn("OpeningPrice", F.col("open"))
        .withColumn("LowestDayPrice", F.col("dayLow"))
        .withColumn("HighestDayPrice", F.col("dayHigh"))
        .withColumn("LowestYearlyPrice", F.col("yearLow"))
        .withColumn("HighestYearlyPrice", F.col("yearHigh"))
        .withColumn("FiftyDayAveragePrice", F.col("fiftyDayAverage"))
        .withColumn("TenDayAverageVolume", F.col("tenDayAverageVolume"))
        .withColumn("ThreeMonthAverageVolume", F.col("threeMonthAverageVolume"))
        .withColumn("TwoHundredDaysAveragePrice", F.col("twoHundredDayAverage"))
        .withColumn("YearOverYearPriceChange", F.col("yearChange"))
        
        # Partycja na podstawie timestamp
        .withColumn("PartitionDate", F.to_date("fetch_timestamp"))
    )

    final_cols = [
        "IndexName",
        "Datetime",
        "CurrentPrice",
        "CurrentVolume",
        "OpeningPrice",
        "LowestDayPrice",
        "HighestDayPrice",
        "LowestYearlyPrice",
        "HighestYearlyPrice",
        "FiftyDayAveragePrice",
        "TwoHundredDaysAveragePrice",
        "TenDayAverageVolume",
        "ThreeMonthAverageVolume",
        "YearOverYearPriceChange",
        "PartitionDate"
    ]

    return res.select(*final_cols)

In [6]:
df_transformed = transform_index_snapshot(df)
print(df_transformed.count())
df_transformed.show(1, vertical=True, truncate=False)

37722
-RECORD 0-----------------------------------------
 IndexName                  | SNP                 
 Datetime                   | 2025-11-20 16:00:48 
 CurrentPrice               | 6737.35009765625    
 CurrentVolume              | 883787038           
 OpeningPrice               | 6737.93017578125    
 LowestDayPrice             | 6737.27978515625    
 HighestDayPrice            | 6770.35009765625    
 LowestYearlyPrice          | 4835.0400390625     
 HighestYearlyPrice         | 6920.33984375       
 FiftyDayAveragePrice       | 6712.009794921875   
 TwoHundredDaysAveragePrice | 6157.763139648438   
 TenDayAverageVolume        | 5352272000          
 ThreeMonthAverageVolume    | 5386517846          
 YearOverYearPriceChange    | 0.11657152556874939 
 PartitionDate              | 2025-11-20          
only showing top 1 row



In [7]:
(df_transformed.write
    .mode("append")
    .format("hive")
    .partitionBy("PartitionDate")
    .saveAsTable("IndexSnapshot"))

25/11/24 16:03:56 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/11/24 16:03:56 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
                                                                                

In [8]:
spark.sql("""SELECT * FROM IndexSnapshot
            LIMIT 10""").show()

+---------+-------------------+----------------+-------------+-------------+----------------+----------------+-----------------+------------------+--------------------+--------------------------+-------------------+-----------------------+-----------------------+-------------+
|IndexName|           Datetime|    CurrentPrice|CurrentVolume| OpeningPrice|  LowestDayPrice| HighestDayPrice|LowestYearlyPrice|HighestYearlyPrice|FiftyDayAveragePrice|TwoHundredDaysAveragePrice|TenDayAverageVolume|ThreeMonthAverageVolume|YearOverYearPriceChange|PartitionDate|
+---------+-------------------+----------------+-------------+-------------+----------------+----------------+-----------------+------------------+--------------------+--------------------------+-------------------+-----------------------+-----------------------+-------------+
|      SNP|2025-11-19 15:00:47| 6671.7099609375|    362266710|6625.83984375|6618.47998046875|6675.14990234375|  4835.0400390625|     6920.33984375|   6709.80739257812

In [9]:
spark.stop()